# Camera calibration from multi-view 2D pose

Three fixed cameras, one moving person, OpenPose BODY_25 keypoints, no
calibration target.  We solve for each camera's focal length and radial
distortion, the pose of cameras 2 and 3 relative to camera 1, and the 3D
position of every joint, by bundle adjustment.

The heavy lifting lives in `camcal.py`; this notebook drives it.

**Two things you need to know before trusting any output:**

1. **The rig is not the same for every subject.**  A model-free epipolar test
   (RANSAC fundamental matrix on the raw correspondences) gives ~33% inliers
   within one subject but only ~13% when four subjects are pooled.  So
   **calibrate per subject/session** -- pooling produces garbage.
2. **Scale is not observable from images.**  No number of cameras fixes it --
   scaling the cameras and the scene together reproduces every pixel exactly.
   It has to come from outside.  We anchor it on a measured distance from
   camera 1 to the point the cameras are aimed at.  **No assumed body size
   enters the solve** (verified in section 5b), so limb lengths come out as an
   output and are used to check the result.

In [1]:
import os
import numpy as np
import pandas as pd

# camcal.py is edited alongside this notebook.  Python caches imported
# modules, so without this a kernel started before an edit keeps the old copy
# and new functions look "missing" (AttributeError on cc.<something>).
%load_ext autoreload
%autoreload 2

import importlib
import camcal
importlib.reload(camcal)     # force a re-read even if already imported
cc = camcal

datapath = '/Volumes/Expansion/MotorDevelopment/Korea/B/Aligned'
# The KEYPOINTS are in 1920x1080 coordinates (verified: x reaches 1922,
# y reaches 1075).  Note the .mp4 files under B_video are 1280x720
# ShanaEncoder re-encodes -- everything below is in 1080p pixel units, so
# scale intrinsics by 2/3 if you ever work off those videos.
IMAGE_SIZE = (1920, 1080)

# The rig used Sony DSC-RX100 bodies.  Focal length is the least
# well-determined parameter in this problem, so pin it with the lens spec.
F_BOUNDS = cc.focal_bounds_for_camera('sony-rx100', IMAGE_SIZE[0])
print('RX100 possible focal range: %.0f - %.0f px (%.0f-%.0f mm equiv)' % (
    F_BOUNDS + tuple(cc.focal_35mm_equiv(np.array(F_BOUNDS)))))

RX100 possible focal range: 1513 - 5396 px (28-101 mm equiv)


## 1. The observation dataframe

Two fixes from the original version:

* `pd.concat(...)` was never assigned, so `validpoints` stayed empty.  Building
  a list and concatenating once is also O(n) instead of O(n^2).
* A joint is missing when its coordinates are **exactly (0, 0)** with
  confidence 0.  A confidence threshold alone lets those through, and a
  (0, 0) point is never a real observation.

In [2]:
def build_points(datapath, prefix='B010'):
    """Flatten the aligned .npz files into one long dataframe."""
    files = sorted(f for f in os.listdir(datapath)
                   if f.startswith(prefix) and f.endswith('.npz')
                   and not f.startswith('.'))     # skip ._ AppleDouble files

    chunks = []
    for bfile in files:
        stub = bfile[:-4]
        user, _, action, rep = stub.split('_')[:4]
        camdata = np.load(os.path.join(datapath, bfile))

        xy = [camdata['xy1'], camdata['xy2'], camdata['xy3']]
        sc = [camdata['score1'], camdata['score2'], camdata['score3']]

        T, J = sc[0].shape                                # J == 25
        frame_idx, joint_idx = np.indices((T, J))

        for cam, (p, s) in enumerate(zip(xy, sc), start=1):
            chunks.append(pd.DataFrame({
                'User': user, 'Action': action, 'Rep': rep, 'Cam': cam,
                'Frame': frame_idx.ravel(),
                'Joint': joint_idx.ravel(),
                'X': p[:, :, 0].ravel(),
                'Y': p[:, :, 1].ravel(),
                'Conf': s.ravel()}))

    allpoints = pd.concat(chunks, ignore_index=True)      # <- assign the result

    # A missing detection is written as exactly (0, 0) with confidence 0.
    allpoints['Valid'] = ((allpoints['Conf'] > 0.5) &
                          ~((allpoints['X'] == 0) & (allpoints['Y'] == 0)))
    return allpoints


allpoints = build_points(datapath, prefix='B010')
print(len(allpoints), 'rows;', f"{100*allpoints['Valid'].mean():.1f}% valid")
allpoints.head()

237600 rows; 91.6% valid


,User,Action,Rep,Cam,Frame,Joint,X,Y,Conf,Valid
0,B010,1,1,1,0,0,1139.590,365.356,0.929988,True
1,B010,1,1,1,0,1,1107.150,430.171,0.873531,True
2,B010,1,1,1,0,2,1048.180,409.643,0.923956,True
3,B010,1,1,1,0,3,998.153,459.594,0.932718,True
4,B010,1,1,1,0,4,951.293,509.660,0.904628,True


## 2. Select frames to calibrate on

Consecutive frames are near-duplicates, so we decimate, keep only frames where
all three views see most of the skeleton, and then pick a subset that **spreads
the subject across each image** at a range of apparent sizes.  That spread is
what constrains focal length -- a subject standing in one spot gives you very
little.

In [3]:
SUBJECT = 'B010'

# No assumed body size enters the solve at any point.  Scale is anchored on a
# measured distance (section 5); limb lengths are initialised from the
# triangulated geometry and are an OUTPUT.  The cohort stature below is used
# ONLY afterwards, by check_stature(), to test the result.
ANCHOR_DISTANCE = 3.0   # m, camera 1 to the point the cameras are aimed at

cand = cc.load_candidates(datapath, subjects=[SUBJECT],
                          conf_thresh=0.6, min_joints=20, stride=6)
obs = cc.select_spread(cand, n_frames=300)

print('frames selected:', obs.xy.shape[0])
print('visible joints per view:', obs.vis.sum(axis=(0, 2)))

scanned 20 files -> 310 candidate frames
frames selected: 300
visible joints per view: [7391 7026 7038]


## 3. Initialise from the data, not from the assumed layout

Seeding the optimiser from a believed layout bakes in whichever way round you
think the cameras were.  If that reading is wrong -- and here it was -- the
solver descends into a mirrored basin and never recovers.

So: take the starting focal length from the RX100 lens spec, recover the 1-2
geometry from the **essential matrix** (`recoverPose` resolves the sign ambiguity by
cheirality, so it cannot return a mirrored rig), scale it with the one number
images can't supply, and solve PnP for camera 3.

In [4]:
# An arbitrary internal scale gauge -- ANY positive value works, because the
# real anchor is applied after the solve (section 5).  Using the layout's
# camera1-camera2 baseline just keeps the numbers readable mid-solve.
# (Corrected layout -- see the note at the bottom on the 45-degree geometry.)
LAYOUT = cc.layout_from_polar([(3.0, 0.0), (1.8, 45.0), (2.5, -45.0)],
                              target_dist=3.0)
BASELINE_12 = float(np.linalg.norm(LAYOUT[1] - LAYOUT[0]))
print('assumed cam1-cam2 baseline:', round(BASELINE_12, 3), 'm')

# Start from the lens spec rather than from apparent subject size: the latter
# needs an assumed height, which is exactly the anthropometric input we are
# trying to avoid.  The wide end is also empirically the best start -- it
# gives the lowest reprojection error of any point in the RX100 range.
f_est = cc.init_focal_from_spec(F_BOUNDS, n_cams=3, where='wide')
print('focal init (px):', np.round(f_est, 1))

cams0 = cc.init_from_essential(obs, f_est, BASELINE_12, image_size=IMAGE_SIZE)

assumed cam1-cam2 baseline: 2.146 m
focal init (px): [1512.7 1512.7 1512.7]
  E(1,2): 3486/7012 inliers, cheirality 3486
  PnP(cam3): 2332/6562 inliers
  centres from data:
    [[ 0.     0.     0.   ]
     [-1.56  -0.228  1.456]
     [ 1.729 -0.064  1.297]]


## 4. Bundle adjustment

Minimise reprojection error over focals, distortion, the poses of cameras 2
and 3, and every 3D point, with two extra constraint families:

* **bone-length constancy** -- each bone holds one length across all frames,
  with left/right sharing a parameter.  This is what stops the reconstruction
  breathing.  It constrains *shape only*: the lengths themselves are free
  parameters initialised from the triangulated geometry, so no assumed size
  enters.
* an optional tether to a limb-length table (`use_bone_prior`), **off by
  default** -- with scale anchored on a measured distance it is not needed,
  and leaving it off keeps limb lengths as an independent check.

Scale itself is pinned **hard** inside `unpack()`.  A global similarity leaves
every reprojection exactly unchanged, so scale is an exact null space of the
data term; a soft prior on it is a handful of residuals against ~13,000 and
loses every time -- the whole reconstruction quietly shrinks.

The stage schedule (`soft_l1` wide -> `soft_l1` with distortion -> `huber`)
matters.  Going straight to a hard robust loss saturates every residual at a
bad starting point, and the optimiser stops on `ftol` having barely moved.

In [5]:
opts = cc.BAOptions(
    fit_k1=False,        # see the distortion note in section 6
    f_bounds=F_BOUNDS,   # the lens cannot go wider than its wide end
    use_bone_prior=False,  # no assumed limb lengths anywhere in the solve
    sigma_bone=0.010,    # m -- bones must hold ONE length across all frames
                         # (left/right share it) -- pure shape, no sizes
    verbose=0,
    max_nfev=200,
)

cams, L, X, res, prob = cc.solve(cams0, obs, opts=opts, sigma_px=4.0)
print('\n', res.message)

[stage 1/3] loss=soft_l1 f_scale=8.0 distortion=off points=7396
   cam1 rms 5.08px  cam2 rms 5.32px  cam3 rms 5.72px   f = [1516. 1812. 1763.]
[stage 2/3] loss=soft_l1 f_scale=4.0 distortion=on points=7396
   cam1 rms 5.07px  cam2 rms 5.32px  cam3 rms 5.81px   f = [1516. 1835. 1765.]
[stage 3/3] loss=huber f_scale=3.0 distortion=on points=7396
   cam1 rms 5.04px  cam2 rms 5.34px  cam3 rms 5.86px   f = [1516. 1836. 1765.]

 `ftol` termination condition is satisfied.


## 5. Metric scale, the floor, and the answers

Scale needs exactly one number from the physical world.  We use the measured
distance from camera 1 to **the point where the three optical axes converge** --
the rig's aim point.  That is better defined than the subject's average
position, because it depends only on where the cameras *point*, so a child
wandering around the capture volume does not move it.  On this data the three
axes converge to within 4-15 cm (about 4% of the anchor distance), and the
child's own distance varies by only ~4% anyway.

Because a global similarity is an exact null direction of the reprojection
cost, rescaling *after* the solve is mathematically identical to constraining
it during -- not an approximation.

Limb lengths are therefore an **output**.  `check_stature()` turns them into an
implied subject height, which is a genuine prediction and the test of whether
the anchor distance actually applies to this session.

Height and tilt are meaningless in camera 1's frame, so we recover a
gravity-aligned frame by RANSAC-fitting a floor plane through the
reconstructed foot keypoints (BODY_25 has toes and heels, joints 19-24).

In [6]:
normal, offset, inliers, footpts = cc.fit_floor(X, prob, obs, thresh=0.04)
print(f'floor plane: {inliers.sum()}/{len(footpts)} foot points on the plane')

# Anchor scale on where the cameras POINT, not on where the child happened to
# stand: the aim point is unaffected by the subject moving around.
cams, X, L, floor, s, info = cc.rescale_to_convergence(
    cams, X, L, prob, normal, offset, distance=ANCHOR_DISTANCE,
    cam=0, mode='ground')

cc.summarise(cams, prob, X, L, floor=floor, layout_prior=LAYOUT)

subj = cc.subject_geometry(X, prob, floor, cams)
print('\ncamera -> subject distance (median):', np.round(subj['dist_median'], 2), 'm')

# Limb lengths are now an OUTPUT, so stature is a real prediction -- and the
# test of whether the anchor distance applies to THIS session.
print()
cc.check_stature(prob, X, L)

spread = cc.anchor_spread(X, prob, floor)
print(f"\nsubject moved: cam1 distance {spread['p10']:.2f}-{spread['p90']:.2f} m "
      f"(sd {spread['sd']:.2f} m = {spread['rel_sd_pct']:.1f}% of the anchor)")

# All three measured distances at once -- the extra two are a free check.
print()
cc.scale_from_distances(cams, X, prob, floor, measured=[3.0, 1.8, 2.5])

floor plane: 1501/1794 foot points on the plane
  aim point at [-0.125  2.632  0.546] in the floor frame
  axes miss it by [0.152 0.12  0.118] m ([5.1 4.  3.9]% of the anchor distance)
  scale: camera 1 -> aim point = 3.00 m (ground), factor 1.138
focal length / field of view
  cam1: f =   1516.4 px   hfov =  64.7 deg   ~28.4 mm equiv   k1 = +0.0000  k2 = +0.0000
  cam2: f =   1835.5 px   hfov =  55.2 deg   ~34.4 mm equiv   k1 = -0.0000  k2 = -0.0000
  cam3: f =   1765.4 px   hfov =  57.1 deg   ~33.1 mm equiv   k1 = +0.0000  k2 = +0.0000

reprojection error (px)
  cam1: rms   5.04  median   3.58  p95    8.93   n = 7389
  cam2: rms   5.34  median   3.22  p95   10.52   n = 7019
  cam3: rms   5.86  median   3.28  p95   10.58   n = 6946

bone lengths (m) -- cv is the diagnostic; want < ~2%
  clavicle    0.132 +- 0.005   cv  3.94%   n = 600
  forearm     0.174 +- 0.008   cv  4.52%   n = 599
  hip_offset  0.086 +- 0.003   cv  3.98%   n = 600
  shank       0.267 +- 0.006   cv  2.24%   n = 600

(0.7920991357376013,
 array([2.57581492, 2.42492625, 2.31757773]),
 array([-0.42418508,  0.62492625, -0.18242227]))

## 5b. Is any body-size assumption still leaking in?

Worth checking rather than asserting.  On B010:

**Swap the `BONE_INIT` table from child to adult** -- a 2x change in every limb
length -- with `use_bone_prior=False`:

| BONE_INIT | thigh | recovered f | heights | implied stature |
|---|---|---|---|---|
| child_3_5 | 0.21 m | 1513 / 1810 / 1752 | 1.28 / 1.22 / 1.24 | 1.242 |
| adult | 0.42 m | 1513 / 1810 / 1752 | 1.28 / 1.22 / 1.24 | 1.242 |

Bit-identical.  No limb-length assumption reaches the solve.  What remains are
bone-length *constancy* and left/right *symmetry*, which constrain shape only
and carry no sizes.

**The one input that did leak** was `estimate_focal_from_scale`, which needs an
assumed subject height to turn apparent pixel size into a starting focal
length.  It is now replaced by `init_focal_from_spec`, which reads the starting
point off the RX100 spec instead.  In practice it never mattered -- any
assumed height at or above ~1 m produced an estimate that clipped to the
lens's wide end anyway -- but the assumption is now gone rather than merely
harmless.

**A real caveat this exposed.**  Sweeping the starting focal across the whole
RX100 range shows the solve is not fully init-independent:

| f start (px) | recovered f | heights | median reproj | stature |
|---|---|---|---|---|
| 1513 | 1513 / 1810 / 1752 | 1.28 / 1.22 / 1.24 | 3.63 / 3.26 / 3.32 | 1.242 |
| 2095 | 1513 / 2050 / 1794 | 1.30 / 1.31 / 1.27 | 3.66 / 3.28 / 3.38 | 1.236 |
| 2872 | 1513 / 2175 / 1810 | 1.33 / 1.42 / 1.27 | 3.68 / 3.34 / 3.40 | 1.236 |
| 5396 | 1513 / 2324 / 1826 | 1.34 / 1.51 / 1.26 | 3.72 / 3.42 / 3.44 | 1.235 |

Camera 1 pins to the wide end from every start, and implied stature is stable
to 0.6%.  But **camera 2's focal spans 1810-2324 px (28%) and its height
1.22-1.51 m (24%)** depending only on where the optimiser starts -- these are
distinct local minima, not a body-size effect.  The wide-end start gives the
lowest reprojection error everywhere, which is why it is the default, but
camera 2 is the least trustworthy parameter in the fit and its height should
be quoted with that spread.

## 6. Checks worth running

**Angles at the subject** are the quantity you actually specified, so check
those rather than the plan coordinates.

In [7]:
plan = floor['plan']
v = plan - subj['centroid'][:2]
v = v / np.linalg.norm(v, axis=1, keepdims=True)
ang = lambda a, b: np.degrees(np.arccos(np.clip(v[a] @ v[b], -1, 1)))
print(f'angle at subject  1-2: {ang(0,1):5.1f} deg')
print(f'angle at subject  1-3: {ang(0,2):5.1f} deg')
print(f'angle at subject  2-3: {ang(1,2):5.1f} deg')
print('you specified      45 / 45 / 90 deg')

angle at subject  1-2:  45.4 deg
angle at subject  1-3:  47.0 deg
angle at subject  2-3:  92.4 deg
you specified      45 / 45 / 90 deg


**Roll** is the strongest free validation available: nothing in the fit forces
it, so tripod-mounted cameras coming out near 0 degrees means the recovered
geometry is real.  Check `floor['roll_deg']` above.

**Distortion.**  `fit_k1=True` drives k1 to its bounds on this data, which is
unphysical -- it is absorbing model error rather than measuring a lens.  Left
off by default.  Turn it on only if you compare median reprojection error both
ways and it genuinely helps.  If you can still get at the cameras, or find any
footage with straight lines in it, calibrating intrinsics properly and fixing
them (`opts.fit_focal = False`) would make all of this far better conditioned.

**Are the correspondences even usable?**  This is model-free and worth running
before believing any fit -- it needs no calibration at all.

In [8]:
import cv2

def epipolar_check(obs, a, b, thresh=3.0):
    m = obs.vis[:, a, :] & obs.vis[:, b, :]
    p1, p2 = obs.xy[:, a][m], obs.xy[:, b][m]
    F, inl = cv2.findFundamentalMat(p1, p2, cv2.FM_RANSAC, thresh, 0.999, 8000)
    inl = inl.ravel().astype(bool)
    p1h = np.hstack([p1, np.ones((len(p1), 1))])
    p2h = np.hstack([p2, np.ones((len(p2), 1))])
    Fx1, Ftx2 = p1h @ F.T, p2h @ F
    num = np.einsum('ij,ij->i', p2h, Fx1) ** 2
    den = Fx1[:, 0]**2 + Fx1[:, 1]**2 + Ftx2[:, 0]**2 + Ftx2[:, 1]**2
    d = np.sqrt(num / np.maximum(den, 1e-12))
    return len(p1), 100 * inl.mean(), float(np.median(d))

for a, b in [(0, 1), (0, 2), (1, 2)]:
    n, pct, med = epipolar_check(obs, a, b)
    print(f'cam{a+1}-cam{b+1}: n={n:6d}  inliers={pct:5.1f}%  median={med:5.2f} px')

# Pool two subjects and watch this collapse -- that is the evidence that the
# rig moved between sessions and must not be calibrated jointly.

cam1-cam2: n=  7012  inliers= 35.5%  median= 2.71 px
cam1-cam3: n=  6939  inliers= 33.3%  median= 2.86 px
cam2-cam3: n=  6569  inliers= 30.3%  median= 3.79 px


## 6b. Focal length is the weak parameter -- pin it with the lens spec

Hold the focal fixed at a range of values and refit everything else.  On B010,
camera 1:

| f (px) | zoom vs wide end | median reproj (px) | cam->subject (m) | cam1 height (m) |
|---|---|---|---|---|
| 1108 | 0.73x | 3.79 / 3.67 / 3.77 | 2.05 / 1.96 / 1.89 | 0.80 |
| 1255 | 0.83x | 3.66 / 3.40 / 3.46 | 2.34 / 2.22 / 2.14 | 0.90 |
| 1403 | 0.93x | 3.60 / 3.30 / 3.35 | 2.62 / 2.49 / 2.39 | 1.01 |
| 1477 | 0.98x | 3.62 / 3.27 / 3.32 | 2.76 / 2.62 / 2.51 | 1.05 |
| 1551 | 1.03x | 3.64 / 3.27 / 3.34 | 2.90 / 2.75 / 2.63 | 1.10 |
| 1699 | 1.12x | 3.72 / 3.35 / 3.41 | 3.18 / 3.02 / 2.88 | 1.19 |
| 1920 | 1.27x | 3.77 / 3.54 / 3.62 | 3.60 / 3.41 / 3.25 | 1.35 |
| 2215 | 1.46x | 3.88 / 3.75 / 3.83 | 4.15 / 3.94 / 3.75 | 1.54 |

Reprojection error barely moves -- 3.6 to 3.9 px across a **2x** range of focal
length -- while the recovered distances swing from 2.0 m to 4.2 m and camera
height from 0.80 m to 1.54 m.  The data cannot separate "narrow lens, far away"
from "wide lens, close up".

This is why the camera spec matters so much.  The RX100's 10.4 mm wide end is
1513 px at 1920 wide, which **excludes the entire lower half of that table** --
every row below 1513 px would need a lens wider than the camera has.  The free
fit put camera 1 at 1477 px, marginally impossible; bounded, it sits on the
wide end at 1514 px.

The RX100 is a zoom (28-100 mm equivalent), so this is a range, not a discrete
set.  To do better:

* If the **original** Sony files survive anywhere, `exiftool -FocalLength
  -FocalLengthIn35mmFormat` on them settles it per clip.  The copies under
  `B_video` are 720p ShanaEncoder re-encodes with all camera metadata stripped,
  so they are no use for this.
* If the operators left the cameras at full wide -- plausible for a fixed rig
  framing a play area -- then `fit_focal=False` with `f = 1513` is defensible
  and removes the ambiguity completely.
* Check whether video mode applies an extra crop; `focal_bounds_for_camera(...,
  video_crop=1.1)` shows what that does (the wide end moves to 1664 px).

## 7. What this actually found (subject B010)

Numbers below are the current output of this notebook, with scale anchored on
camera 1 being 3.00 m from the aim point.

| quantity | recovered | you assumed |
|---|---|---|
| angles at the subject | **45.4 / 47.0 / 92.4 deg** | 45 / 45 / 90 |
| focal length | **1516 / 1836 / 1765 px** (28.4 / 34.4 / 33.1 mm equiv) | unknown |
| camera height | **1.29 / 1.26 / 1.22 m** | unknown |
| downward tilt | **11.5 / 13.6 / 12.2 deg** | unknown |
| roll | **1.6 / -0.7 / 0.6 deg** | -- |
| camera -> child (median) | **3.35 / 3.14 / 3.01 m** | 3.0 / 1.8 / 2.5 |
| median reprojection | **3.6 / 3.2 / 3.3 px** | -- |

**What is solid.**  The angles match your 45/45/90 to within 2 degrees, and
nothing in the fit forced them.  Roll comes out within 1.6 degrees of zero on
all three cameras -- also unforced, and exactly what tripod-mounted cameras
must give.  Camera 1's focal pins to 1516 px, the RX100's wide end, from every
starting point tried.  These are all scale-invariant or spec-anchored, so they
do not depend on the anchor distance.

**What is not.**  The implied stature is **1.24 m**, too tall for a 3-4.5 year
old (expect 0.90-1.15).  Since limb lengths are an output here, that is a
direct signal that **camera 1 was not 3.00 m from the aim point in this
session** -- everything metric is inflated by roughly 20%.  Scaling to a 1.02 m
child instead gives ~2.8 / 2.6 / 2.5 m and heights near 1.05 m.

The three measured distances are mutually inconsistent with the recovered
shape.  Best single scale over all three leaves:

| | implied | measured | residual |
|---|---|---|---|
| camera 1 | 2.58 m | 3.0 m | -0.42 |
| camera 2 | 2.42 m | 1.8 m | **+0.62** |
| camera 3 | 2.32 m | 2.5 m | -0.18 |

Camera 2 is the odd one out by a wide margin.  The reconstruction fixes the
*ratios* between these distances independently of scale, so at least one
remembered distance is wrong -- most likely camera 2's 1.8 m.

Movement is not the problem: the child's distance to camera 1 spans only
3.05-3.37 m (sd 4.3% of the anchor).

## 8. Calibrate every session

The rig is **not** the same across subjects, so loop.  Two guards:

* sessions with too little usable data raise `InsufficientData` rather than
  returning a confident-looking fit -- B013 and B014 have only 4-8% of frames
  with the child well seen by all three cameras;
* `check_stature()` flags sessions where the implied child height is
  implausible, which means the 3 m anchor does not apply there.

Running B010-B014:

| subject | median reproj (px) | heights (m) | tilt (deg) | implied stature |
|---|---|---|---|---|
| B010 | 3.6 / 3.3 / 3.3 | 1.28 / 1.22 / 1.24 | 11.4 / 13.0 / 12.6 | 1.24 -- **check** |
| B011 | 3.5 / 4.3 / 4.3 | 0.86 / 0.84 / 0.94 | 3.2 / 6.5 / 9.8 | 0.94 -- ok |
| B012 | 4.0 / 5.4 / 5.8 | 0.74 / 0.63 / 0.69 | 6.9 / 9.7 / 14.8 | 0.73 -- **check** |
| B013 | -- | -- | -- | skipped, 18 frames |
| B014 | -- | -- | -- | skipped, 34 frames |

Only B011 is consistent with a 3 m camera-1 distance.  Your tape measure was
taken once and cannot apply to all 101 sessions.  For the rest, anchoring on
cohort-mean stature (1.02 m, individual spread ~6%) is more defensible than a
3 m that can be 25% wrong -- swap `rescale_to_convergence` for
`rescale_to_anthropometry(..., target=cc.bone_lengths_for(1.02))` there.

In [10]:
def calibrate_session(subject, n_frames=250, fit_k1=False, verbose=False):
    cand = cc.load_candidates(datapath, subjects=[subject], conf_thresh=0.6,
                              min_joints=20, stride=6, verbose=False)
    obs = cc.select_spread(cand, n_frames=n_frames)

    f_est = cc.init_focal_from_spec(F_BOUNDS, n_cams=3, where='wide')
    cams0 = cc.init_from_essential(obs, f_est, BASELINE_12,
                                   image_size=IMAGE_SIZE, verbose=verbose)

    opts = cc.BAOptions(fit_k1=fit_k1, f_bounds=F_BOUNDS,
                        use_bone_prior=False, verbose=0, max_nfev=200)
    cams, L, X, res, prob = cc.solve(cams0, obs, opts=opts, sigma_px=4.0,
                                     verbose=verbose)
    normal, offset, inl, _ = cc.fit_floor(X, prob, obs, thresh=0.04)
    cams, X, L, floor, s, info = cc.rescale_to_convergence(
        cams, X, L, prob, normal, offset, ANCHOR_DISTANCE, 0, 'ground',
        verbose=False)
    stat = cc.check_stature(prob, X, L, verbose=False)
    subj = cc.subject_geometry(X, prob, floor, cams)
    err = cc.reprojection_errors(cams, prob, X)

    return dict(subject=subject, n_frames=obs.xy.shape[0], scale=s,
                stature=stat,
                cams=cams, floor=floor, subject_geom=subj, bones=cc.bone_stats(prob, X, L),
                median_px=[err[c]['median'] for c in sorted(err)])


subjects = sorted({f.split('_')[0] for f in os.listdir(datapath)
                   if f.endswith('.npz') and not f.startswith('.')})

results, skipped = {}, {}
for sub in subjects:                       # drop the slice to do all 101
    try:
        results[sub] = calibrate_session(sub)
        r = results[sub]
        ok = 'ok ' if r['stature']['plausible'] else 'CHECK'
        print(f"{sub}: {ok} median {np.round(r['median_px'],2)} px  "
              f"height {np.round(r['floor']['height'],2)}  "
              f"tilt {np.round(r['floor']['tilt_deg'],1)}  "
              f"stature {r['stature']['estimate']:.2f} m")
    except cc.InsufficientData as e:
        skipped[sub] = str(e)
        print(f'{sub}: SKIPPED -- {str(e).split(chr(46))[0]}')

print(f'\ncalibrated {len(results)}, skipped {len(skipped)}')

B010: CHECK median [3.63 3.26 3.32] px  height [1.28 1.22 1.24]  tilt [11.4 13.  12.6]  stature 1.24 m
B011: ok  median [3.48 4.28 4.26] px  height [0.86 0.84 0.94]  tilt [3.2 6.5 9.8]  stature 0.94 m
B012: CHECK median [4.   5.4  5.75] px  height [0.74 0.63 0.69]  tilt [ 6.9  9.7 14.8]  stature 0.73 m
B013: SKIPPED -- only 18 usable frames (need >= 80)
B014: SKIPPED -- only 34 usable frames (need >= 80)
B015: SKIPPED -- only 68 usable frames (need >= 80)
B016: CHECK median [11.98 38.48 22.4 ] px  height [2.16 0.61 0.67]  tilt [32.7 14.6  9.8]  stature 0.38 m
B019: CHECK median [4.09 3.39 4.62] px  height [1.12 0.91 0.99]  tilt [10.8 18.6 10.4]  stature 0.82 m
B020: ok  median [4.51 5.62 7.91] px  height [0.87 1.12 0.77]  tilt [ 4.7 10.9  5.7]  stature 1.07 m
B021: SKIPPED -- only 8 usable frames (need >= 80)
B022: CHECK median [13.   14.39  7.93] px  height [0.78 0.73 0.85]  tilt [11.  13.5 14. ]  stature 0.63 m
B023: CHECK median [3.73 3.4  3.7 ] px  height [1.05 1.02 1.1 ]  tilt [2.

RuntimeError: no candidate frames survived filtering